In [ ]:
import re
from pathlib import Path

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, classification_report,
 )
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Dataset ALEX (Ss_CODE)

## S/ SMOTE

In [ ]:
# -------- 1) Carregar dados preparados (sem descartar linhas) --------
PREP_CANDIDATES = [
    Path('artifacts') / 'prepared' / 'alex_prepared.joblib',
    Path('Projeto') / 'artifacts' / 'prepared' / 'alex_prepared.joblib',
]
PREP_PATH = next((p for p in PREP_CANDIDATES if p.exists()), None)
if PREP_PATH is None:
    tried = ', '.join(str(p) for p in PREP_CANDIDATES)
    raise FileNotFoundError(f'Não encontrei alex_prepared.joblib. Tentei: {tried}')

bundle = joblib.load(PREP_PATH)
X_all = pd.DataFrame(bundle['X_values'], columns=bundle['feature_cols'])
y_all = np.asarray(bundle['y_enc'], dtype=int)

le = LabelEncoder()
le.classes_ = np.asarray(bundle.get('label_classes', []), dtype=object)

TARGET_COL = bundle.get('target_col', 'Target (Ss_CODE)')
FEATURE_COLS = bundle.get('feature_cols', [])

print('PREP_PATH:', PREP_PATH.resolve())
print('total rows:', int(bundle.get('n_rows', len(X_all))))
print('invalid target rows:', int((y_all == -1).sum()))
print('X_all shape:', X_all.shape)

raw shape: (80646, 19)


,[I] Family and Type,[I] Type,[I] Workset,[I] Category,[I] IfcGUID,[I] Level,[I] Mark,[T] Type IfcGUID,[T] Workset,[T] Family Name,[I] Construction Block,[T] Classification Number,[T] Type Name,[T] Category,[I] Type Id,[I] Volume,Target (EF_CODE),Target (Ss_CODE),Target (Pr_CODE)
0,TPF-ValveBox.Xylem-Standard-GEOM: Standard,Standard,DRG-Exterior,Generic Models,082Y9U8n57RxOQ13jCCxiV,Ground Floor,NaN,082Y9U8n57RxOQ13jCCxXM,Family : Generic Models : TPF-ValveBox.Xylem-...,TPF-ValveBox.Xylem-Standard-GEOM,NaN,NaN,Standard,Generic Models,1731873.0,0.04 m³,0,0,0
1,TPF-Wastewater-Contaminated-Station.Flygt-Auto...,Standard,DRG-Exterior,Generic Models,082Y9U8n57RxOQ13jCCxWl,Ground Floor,NaN,082Y9U8n57RxOQ13jCCx7K,Family : Generic Models : TPF-Wastewater-Cont...,TPF-Wastewater-Contaminated-Station.Flygt-Auto...,NaN,NaN,Standard,Generic Models,1729699.0,0.76 m³,0,0,0
2,TPF-InspectionBox-Blind-RefLevels: 0.60 x 0.60,0.60 x 0.60,DRG,Mechanical Equipment,1uIinGoWPC5RVu5GeQ3hP0,Ground Floor,CVO3C01.3,3BOmqxrYf5hxpVbPIqK0b6,Family : Mechanical Equipment : TPF-Inspectio...,TPF-InspectionBox-Blind-RefLevels,O3,NaN,0.60 x 0.60,Mechanical Equipment,1744703.0,0.85 m³,0,0,0


In [ ]:
# -------- 2) Construir X e y para treino (sem apagar dados) --------
MIN_SAMPLES_PER_CLASS = 6  # referência (SMOTE-safe)

In [ ]:
# -------- 3) Subset treinável (labels válidos + split possível) --------
mask_labeled = y_all >= 0
X_labeled = X_all.loc[mask_labeled].copy()
y_labeled = y_all[mask_labeled]

counts = pd.Series(y_labeled).value_counts()
ok_classes = counts[counts >= 2].index
mask_ok = pd.Series(y_labeled).isin(ok_classes).to_numpy()
X = X_labeled.loc[mask_ok].copy()
y = y_labeled[mask_ok]

dropped_too_small = counts[counts < 2]
print('trainable rows:', len(X))
print('n_classes trainable:', int(pd.Series(y).nunique()))
if len(dropped_too_small) > 0:
    print('classes excluídas do treino (count<2):')
    print(dropped_too_small)

print('X shape:', X.shape, 'y shape:', y.shape)
print('classes:', int(pd.Series(y).nunique()))

after filtering shape: (27872, 13)
n_classes: 34
min class count: 6
X shape: (27872, 697) y shape: (27872,)
classes: 34


In [24]:
# -------- 4) Train/Test Split --------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

In [25]:
# -------- 1) SVM sem SMOTE (usa class_weight balanced) --------
svm_no = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", class_weight="balanced", random_state=42)
)
svm_no.fit(X_train, y_train)
pred_no = svm_no.predict(X_test)

acc_no = accuracy_score(y_test, pred_no)
bal_no = balanced_accuracy_score(y_test, pred_no)
f1m_no = f1_score(y_test, pred_no, average="macro")
f1w_no = f1_score(y_test, pred_no, average="weighted")

print("\n=== TESTE — SVM SEM SMOTE ===")
print(f"Accuracy:         {acc_no:.4f}")
print(f"Balanced Accuracy:{bal_no:.4f}")
print(f"F1-macro:         {f1m_no:.4f}")
print(f"F1-weighted:      {f1w_no:.4f}")
print("\nClassification report:")
print(classification_report(y_test, pred_no, digits=4))


=== TESTE — SVM SEM SMOTE ===
Accuracy:         0.9993
Balanced Accuracy:0.9975
F1-macro:         0.9973
F1-weighted:      0.9993

Classification report:
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        47
           1     1.0000    1.0000    1.0000        13
           2     1.0000    1.0000    1.0000       128
           3     1.0000    1.0000    1.0000        77
           4     1.0000    1.0000    1.0000        54
           5     1.0000    0.9231    0.9600        13
           6     1.0000    1.0000    1.0000         2
           7     1.0000    1.0000    1.0000       118
           8     1.0000    1.0000    1.0000       513
           9     1.0000    1.0000    1.0000       415
          10     1.0000    1.0000    1.0000      1046
          11     1.0000    1.0000    1.0000       126
          12     1.0000    1.0000    1.0000         1
          13     1.0000    1.0000    1.0000       110
          14     1.0000    1.0000 

In [26]:
# -------- 2) SVM com SMOTE --------

# Em multi-classe, algumas classes podem ficar com poucos exemplos no treino
# (mesmo que no dataset completo tenham >= MIN_SAMPLES_PER_CLASS).
# Por isso ajustamos k_neighbors dinamicamente para evitar o erro:
# "Expected n_neighbors <= n_samples_fit".
min_class_train = int(np.bincount(y_train).min())
if min_class_train < 2:
    raise ValueError('SMOTE não é possível: há classes com <2 amostras no treino.')

k_neighbors = min(5, min_class_train - 1)

svm_sm = ImbPipeline(steps=[
    ("smote", SMOTE(random_state=42, k_neighbors=k_neighbors, sampling_strategy='not majority')),
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", random_state=42))
])
svm_sm.fit(X_train, y_train)
pred_sm = svm_sm.predict(X_test)

acc_sm = accuracy_score(y_test, pred_sm)
bal_sm = balanced_accuracy_score(y_test, pred_sm)
f1m_sm = f1_score(y_test, pred_sm, average="macro")
f1w_sm = f1_score(y_test, pred_sm, average="weighted")

print("\n=== TESTE — SVM COM SMOTE ===")
print(f"SMOTE k_neighbors: {k_neighbors}")
print(f"Accuracy:         {acc_sm:.4f}")
print(f"Balanced Accuracy:{bal_sm:.4f}")
print(f"F1-macro:         {f1m_sm:.4f}")
print(f"F1-weighted:      {f1w_sm:.4f}")
print("\nClassification report:")
print(classification_report(y_test, pred_sm, digits=4))


=== TESTE — SVM COM SMOTE ===
SMOTE k_neighbors: 4
Accuracy:         0.9996
Balanced Accuracy:0.9998
F1-macro:         0.9992
F1-weighted:      0.9996

Classification report:
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        47
           1     1.0000    1.0000    1.0000        13
           2     1.0000    1.0000    1.0000       128
           3     1.0000    1.0000    1.0000        77
           4     1.0000    1.0000    1.0000        54
           5     1.0000    1.0000    1.0000        13
           6     1.0000    1.0000    1.0000         2
           7     1.0000    1.0000    1.0000       118
           8     1.0000    1.0000    1.0000       513
           9     1.0000    1.0000    1.0000       415
          10     1.0000    1.0000    1.0000      1046
          11     1.0000    1.0000    1.0000       126
          12     1.0000    1.0000    1.0000         1
          13     1.0000    1.0000    1.0000       110
          14 

Sem SMOTE o SVM “força” muito recall nas classes pequenas (ex.: 4 e 5), mas com precisão baixíssima → muitos falsos positivos.

Com SMOTE sobe bastante o accuracy e o f1_weighted, e melhora a macro, mas ainda fica bem abaixo do RF.

In [27]:
# -------- 3) Validação Cruzada (para robustez) --------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scorings = {
    "acc": "accuracy",
    "bal_acc": "balanced_accuracy",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted"
}

# CV sem SMOTE
cv_no = cross_validate(svm_no, X, y, cv=cv, scoring=scorings, n_jobs=-1)

# CV com SMOTE
# Em CV não dá para escolher k_neighbors dinamicamente por fold com este setup;
# usamos k_neighbors pequeno para evitar falhas em classes raras nos folds.
svm_sm_cv = ImbPipeline(steps=[
    ("smote", SMOTE(random_state=42, k_neighbors=1, sampling_strategy='not majority')),
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", random_state=42))
])
cv_sm = cross_validate(svm_sm_cv, X, y, cv=cv, scoring=scorings, n_jobs=-1)

def resumo_cv(nome, res):
    print(f"\n=== CV (5-fold) — {nome} ===")
    for k in scorings.keys():
        vals = res[f"test_{k}"]
        print(f"{k:12s}: {vals.mean():.4f} ± {vals.std():.4f} "
              f"(min={vals.min():.4f}, max={vals.max():.4f})")

resumo_cv("SVM SEM SMOTE", cv_no)
resumo_cv("SVM COM SMOTE", cv_sm)


=== CV (5-fold) — SVM SEM SMOTE ===
acc         : 0.9985 ± 0.0005 (min=0.9978, max=0.9993)
bal_acc     : 0.9733 ± 0.0071 (min=0.9660, max=0.9849)
f1_macro    : 0.9748 ± 0.0095 (min=0.9664, max=0.9888)
f1_weighted : 0.9984 ± 0.0004 (min=0.9979, max=0.9992)

=== CV (5-fold) — SVM COM SMOTE ===
acc         : 0.9992 ± 0.0005 (min=0.9984, max=0.9996)
bal_acc     : 0.9850 ± 0.0129 (min=0.9683, max=0.9986)
f1_macro    : 0.9856 ± 0.0136 (min=0.9680, max=0.9988)
f1_weighted : 0.9991 ± 0.0005 (min=0.9984, max=0.9996)


In [28]:
# --- 3) Função utilitária para Permutation Importance ---
def compute_perm_importance(model, X_test, y_test, feature_names, scoring="balanced_accuracy",
                            n_repeats=10, random_state=42, n_jobs=-1, top_n=15, title_suffix=""):
    """
    Calcula permutation importance no conjunto de TESTE.
    Retorna DataFrame ordenado e mostra gráfico das top N features.
    """
    print(f"\nCalculando Permutation Importance ({title_suffix}) ...")
    result = permutation_importance(
        model, X_test, y_test,
        scoring=scoring,
        n_repeats=n_repeats,
        random_state=random_state,
        n_jobs=n_jobs
    )
    imp_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance_mean": result.importances_mean,
        "Importance_std": result.importances_std
    }).sort_values("Importance_mean", ascending=False)

    print(imp_df.head(top_n))

    # Gráfico Top N
    top = imp_df.head(top_n).iloc[::-1]  # inverter para barra horizontal bonita
    plt.figure(figsize=(9, max(5, 0.35*len(top))))
    plt.barh(top["Feature"], top["Importance_mean"], xerr=top["Importance_std"])
    plt.xlabel(f"Permutation importance (média Δ {scoring})")
    plt.title(f"Permutation Importance — SVM RBF {title_suffix}")
    plt.tight_layout()
    plt.show()

    return imp_df


In [ ]:
# --- 4) Rodar Permutation Importance (balanced_accuracy) ---
imp_no   = compute_perm_importance(
    svm_no, X_test, y_test, feature_names=X.columns,
    scoring="balanced_accuracy", n_repeats=10, title_suffix="(SEM SMOTE)"
)

imp_sm   = compute_perm_importance(
    svm_sm, X_test, y_test, feature_names=X.columns,
    scoring="balanced_accuracy", n_repeats=10, title_suffix="(COM SMOTE)"
)


Calculando Permutation Importance ((SEM SMOTE)) ...
